# Iridium 1.0 -- TPU v5e-1 builder (Colab/Kaggle)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sporadicstudiosind-cloud/test/blob/claude/gallant-faraday-lhycva/notebooks/train_iridium_tpu_colab.ipynb)

**The model is untrained until you run the training cell, and that cell is
gated behind an explicit confirmation below** -- a free TPU session is
quota you cannot get back, and this notebook would rather make you say so
than start on autopilot.

This targets the free **TPU v5e-1** runtime (`torch_xla`), not a GPU. It
starts from a preset like the other studio notebooks, then exposes the raw
geometry -- core layers, superstack layers, superstack count -- as
overrides, because a TPU's extra memory (48 GB host RAM on a Colab v5e-1)
is exactly the room to try a deeper or wider variant of a preset before it
has a name.

Needs network and the `datasets` package for the same reason as the other
notebooks: the text/chat/tokenizer path streams real data and trains a
subword tokenizer from it.

## 1. Get the source (private repo)

In [ ]:
# This repository is private until Iridium 1.0 ships. This cell never
# hard-codes or prints a token: it reads one from the host's own secret
# store (or an env var on a plain Jupyter host) and hands it to git only
# through an environment-scoped header, so it never lands in .git/config.
import os, sys, subprocess
from pathlib import Path
token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None

REPO_OWNER = 'sporadicstudiosind-cloud'
REPO_NAME = 'test'
RELEASE_BRANCH = 'claude/gallant-faraday-lhycva'
ROOT = Path.cwd()

if (ROOT / 'iridium' / 'presets.py').is_file():
    print('Already inside a checkout of the repository:', ROOT)
else:
    ROOT = Path.cwd() / REPO_NAME
    if ROOT.is_dir():
        print('Reusing existing checkout at', ROOT)
    elif token:
        # The token travels in an environment-scoped git config header:
        # never in the URL (git would store it in .git/config) and never in
        # the argv (a failed subprocess prints its argv).
        import base64
        basic = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
        env = dict(os.environ, GIT_CONFIG_COUNT='1',
                   GIT_CONFIG_KEY_0='http.https://github.com/.extraheader',
                   GIT_CONFIG_VALUE_0=f'AUTHORIZATION: basic {basic}',
                   GIT_TERMINAL_PROMPT='0')
        done = subprocess.run(['git', 'clone', '--depth', '1', '--branch', RELEASE_BRANCH,
                               f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git',
                               str(ROOT)], env=env, capture_output=True, text=True)
        del env, basic
        if done.returncode != 0:
            raise RuntimeError('git clone failed (exit %d). Check the token can read '
                               'the repository and the branch exists.' % done.returncode)
        print('Cloned', REPO_OWNER + '/' + REPO_NAME, '@', RELEASE_BRANCH, 'to', ROOT)
    else:
        print('No GITHUB_TOKEN found. In Colab: the key icon in the left sidebar, Secrets, add GITHUB_TOKEN with a fine-grained PAT that can read this repository, then toggle notebook access on.')
        print('Trying an unauthenticated clone (works only if the repo is public)...')
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', RELEASE_BRANCH,
                       f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git', str(ROOT)],
                      check=True)
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('working directory:', Path.cwd())

## 2. Install (adds torch_xla for the TPU runtime)

In [ ]:
!pip install -q -e . 2>/dev/null || pip install -q torch numpy datasets pyyaml psutil
!pip install -q torch_xla[tpu] -f https://storage.googleapis.com/libtpu-releases/index.html

## 3. Detect the TPU and pick a starting preset

In [ ]:
import torch
from iridium.presets import PRESETS, FREE_TIERS, get_preset, preset_table
from iridium.runtime.device import detect

try:
    import torch_xla.core.xla_model as xm
    DEVICE = str(xm.xla_device())
    print('XLA device:', DEVICE)
except Exception as exc:
    print('torch_xla is not usable here (', exc, '); falling back to CPU/GPU detection.')
    DEVICE = detect().device

PRESET = 'chat-100m'  # a TPU v5e-1's 16 GB and 48 GB host RAM affords more than chat-34m
preset = get_preset(PRESET)
print()
print(preset_table())

## 4. Geometry overrides

Leave any override at `None` to keep the preset's own value. These change
`preset.config.core`/`preset.config.stacks` directly; the dry-run cell
below re-derives the parameter count from the *overridden* shapes, so a
mismatch here is caught before training, not after.

In [ ]:
from dataclasses import replace

CORE_LAYERS = None         # int, overrides preset.config.core.n_layers
SUPERSTACK_LAYERS = None    # int, overrides preset.config.stacks.n_layers
SUPERSTACK_COUNT = None     # int, overrides preset.config.stacks.n_stacks

cfg = preset.config
new_core = cfg.core if CORE_LAYERS is None else replace(cfg.core, n_layers=CORE_LAYERS)
new_stacks = cfg.stacks
if SUPERSTACK_LAYERS is not None:
    new_stacks = replace(new_stacks, n_layers=SUPERSTACK_LAYERS)
if SUPERSTACK_COUNT is not None:
    new_stacks = replace(new_stacks, n_stacks=SUPERSTACK_COUNT)
cfg = replace(cfg, core=new_core, stacks=new_stacks)
preset = replace(preset, config=cfg)
print(preset.config.report().render())

## 5. Dry run -- build, cost, audit

In [ ]:
from iridium.training.run_preset import dry_run

dry_run_result = dry_run(preset)
if not dry_run_result['match']:
    raise RuntimeError(
        f"parameter count mismatch: built {dry_run_result['parameters_built']:,}, "
        f"formula says {dry_run_result['parameters_formula']:,}. Do not train "
        "until this reconciles -- report it rather than proceeding."
    )

## 6. Confirm and train

`torch_xla` compiles static shapes; this architecture's router dispatches
a *variable* number of tokens per superstack per step, which forces a
recompile per shape (or padding to a fixed capacity) on a real TPU core.
That is a real consequence of dynamic routing on XLA, not a missing
feature here -- expect the first several steps to be slow while XLA traces
shapes, and expect a shape change (a new batch composition) to trigger
another trace.

Set `CONFIRM_TRAIN = True` once the dry run above looks right. This is the
gate: nothing above this cell spends TPU quota.

In [ ]:
from pathlib import Path
from iridium.training.run_preset import train_preset

CONFIRM_TRAIN = False
OUT_DIR = Path('/content/drive/MyDrive/iridium-runs') if Path('/content/drive').is_dir() else Path('runs')
RESUME_FROM = ''

if not CONFIRM_TRAIN:
    print('CONFIRM_TRAIN is False; not training. Set it to True and rerun this cell.')
    checkpoint = None
else:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    checkpoint = train_preset(
        preset, device=DEVICE, out=str(OUT_DIR), resume=(RESUME_FROM or None), seed=0,
    )
    print('final checkpoint:', checkpoint)

## 7. Chat with the result

In [ ]:
from iridium.training.trainer import load_checkpoint
from iridium.runtime.chat import ChatSession

if checkpoint is None:
    print('No checkpoint yet -- confirm and train in section 6 first.')
else:
    model, manifest = load_checkpoint(str(checkpoint), device='cpu')
    chat = ChatSession(model)
    print(chat.send('Hello! What are you, and what can you actually do right now?'))